# Sequentially build database
We will use the canonical peptides and the C-terminal isoform peptides as background and then sequentially add the internal and N-terminal peptides. Sequentially means that we will have a new mapping excluding all the C-terminal isoforms and from that mapping we'll go through the isoforms one by one and add them if they are unique to what has been added to the databse up to that point.

# Setup
## Import and install Packages

In [ ]:
import sys
import os
import session_info

# Add the '0_functions' folder to sys.path
sys.path.append(os.path.join(os.getcwd(), '..', '00_functions'))

In [ ]:
import pandas as pd
import ast

from Bio import SeqIO
from Bio.Seq import Seq
from Bio.SeqRecord import SeqRecord

In [ ]:
# Display session information
session_info.show()

## Set working directory

In [ ]:
cwd = os.getcwd()

if not cwd.endswith("Isoform_Database/Jupyter_environment/Isoform_Database_SIAF/02_Isoform_Database"):
    print("WARNING: The working directory is not set to the '02_Isoform_Database'!")
    print("Current working directory:", cwd)
else:
    print(f"Working directory is correctly set to the '02_Isoform_Database' folder (\"{cwd}\").")

In [ ]:
# Data directories
digestion_output_dir = "data/digestion_output"
fasta_dir = "../01_UniProt/data/raw_data/fasta"
mapping_dir = "../01_UniProt/data/mapping/"

## Read in dataframes

In [ ]:
iso_canonical_mapping = pd.read_csv(os.path.join(mapping_dir, 'iso_canonical_mapping.csv'))
iso_canonical_mapping_flat = pd.read_csv(os.path.join(mapping_dir, 'iso_canonical_mapping_flat.csv'))

digestion_Trypsin_filtered =  pd.read_csv(os.path.join(digestion_output_dir, 'digestion_Trypsin_filtered.csv'))
digestion_TrypsinP_filtered =  pd.read_csv(os.path.join(digestion_output_dir, 'digestion_TrypsinP_filtered.csv'))

df_c_terminal_try = pd.read_csv("data/df_c_terminal_try.csv")
df_c_terminal_tryP = pd.read_csv("data/df_c_terminal_tryP.csv")

# Find unique peptides for internal and N-terminal isoforms
Sequential implementation: The function looks at Isoform 1 for every single canpnical protein in the mapping. Only after every protein has had a chance to claim its first marker it moves on to look at the next Isoform etc. I checked wether it makes a difference to check for duplicates in the peptides added in each round and we end up with the same amount of unique peptides with and without checking in the round. I still kept the chekc because if there were a duplicate in the round it wouldn't be uniqe to either of these peptides.

In [ ]:
def find_round_robin_strict(digestion_df, mapping_df, all_isoform_ids, c_term_df):
    # 1. Initial Forbidden Set (Background Proteome + C-term Priority)
    background_proteome = digestion_df[~digestion_df['ID'].isin(all_isoform_ids)]
    forbidden_peptides = set(background_proteome['seq'].unique())

    c_term_ids = set(c_term_df['Isoform_ID'])
    c_term_all_peps = digestion_df[digestion_df['ID'].isin(c_term_ids)]
    forbidden_peptides.update(c_term_all_peps['seq'].unique())

    # Pre-process peptides for quick lookup
    pep_dict = digestion_df.groupby('ID')['seq'].apply(set).to_dict()
    
    # Prepare mapping (canonical -> list of internal isoforms)
    internal_map = []
    for _, row in mapping_df.iterrows():
        iso_list = row['Isoforms']
        if isinstance(iso_list, str): iso_list = ast.literal_eval(iso_list)
        filtered_list = [i for i in iso_list if i not in c_term_ids]
        if filtered_list: internal_map.append(filtered_list)

    max_rounds = max(len(lst) for lst in internal_map)
    final_results = []

    # 4. Round-Robin Loop
    for round_idx in range(max_rounds):
        # Step A: Identify potential unique peptides for every isoform in this round
        round_candidates = {}
        peptide_collision_tracker = {} # To count how many isoforms in this round share a peptide

        for protein_isoforms in internal_map:
            if round_idx < len(protein_isoforms):
                iso_id = protein_isoforms[round_idx]
                current_iso_peptides = pep_dict.get(iso_id, set())
                
                # Must not be in background, C-terms, or previous rounds
                potential_peptides = current_iso_peptides - forbidden_peptides
                
                if potential_peptides:
                    round_candidates[iso_id] = potential_peptides
                    # Track collisions within THIS round
                    for pep in potential_peptides:
                        peptide_collision_tracker[pep] = peptide_collision_tracker.get(pep, 0) + 1

        # Step B: Filter out peptides that collided within this round
        round_final_forbidden_update = set()
        
        for protein_isoforms in internal_map:
            if round_idx < len(protein_isoforms):
                iso_id = protein_isoforms[round_idx]
                
                # Check if we even had candidates
                iso_potentials = round_candidates.get(iso_id, set())
                
                # Strictly unique: peptide count in this round must be exactly 1
                strictly_unique = {p for p in iso_potentials if peptide_collision_tracker[p] == 1}
                
                if strictly_unique:
                    best_peptide = max(strictly_unique, key=len)
                    status = "Unique Marker Found"
                else:
                    best_peptide = None
                    status = "Shared peptide"

                final_results.append({
                    "Isoform_ID": iso_id,
                    "Round": round_idx + 1,
                    "Unique_Peptide": best_peptide,
                    "Status": status
                })
                
                # Add ALL of this isoform's peptides to the forbidden list for Round N+1
                round_final_forbidden_update.update(pep_dict.get(iso_id, set()))

        # Update forbidden set once the whole round is validated
        forbidden_peptides.update(round_final_forbidden_update)

    return pd.DataFrame(final_results)

# --- EXECUTION ---

# Define set of all isoform IDs 
all_isoform_ids = set(iso_canonical_mapping_flat['Isoform_ID'].unique())

# Generate two dataframes for Try and TryP
sequential_results_Try = find_round_robin_strict(digestion_Trypsin_filtered, iso_canonical_mapping, all_isoform_ids, df_c_terminal_try)
sequential_results_TryP = find_round_robin_strict(digestion_TrypsinP_filtered, iso_canonical_mapping, all_isoform_ids, df_c_terminal_tryP)

In [ ]:
sequential_results_Try

In [ ]:
print(f"--- Summary Statistics Try ---")
print(sequential_results_Try['Status'].value_counts())

In [ ]:
print(f"--- Summary Statistics TryP ---")
print(sequential_results_TryP['Status'].value_counts())

## Build databse fasta file
Put C-terminal isoforms peptides (only C-terminus) and internal/N-terminal isoform peptides (only tryptic peptide) into a fasta file -> isoform database

In [ ]:
# Trypsin results
final_records = []

# 1. C-terminal Isoforms (Filtering for valid peptides)
for _, row in df_c_terminal_try.iterrows():
    iso_id = row['Isoform_ID']
    pep_seq = row['Peptide']
    
    rec = SeqRecord(Seq(pep_seq), id=f"sp|{iso_id}|CTERM_VAR", description="")
    final_records.append(rec)

# 2. Add Internal/N-terminal Isoforms (Filtering for Status)
for _, row in sequential_results_Try.iterrows():
    if row['Status'] == "Unique Marker Found":
        pep_seq = row['Unique_Peptide']
        iso_id = row['Isoform_ID']
        
        if pd.notna(pep_seq):
            rec = SeqRecord(Seq(pep_seq), id=f"sp|{iso_id}|INTERNAL_VAR", description="")
            final_records.append(rec)

# Write to file
with open("Try_Isoform_Database_Final.fasta", "w") as output_handle:
    SeqIO.write(final_records, output_handle, "fasta")

print(f"FASTA created with {len(final_records)} entries.")

In [ ]:
# Trypsin results
final_records = []

# 1. C-terminal Isoforms (Filtering for valid peptides)
for _, row in df_c_terminal_tryP.iterrows():
    iso_id = row['Isoform_ID']
    pep_seq = row['Peptide']
    
    rec = SeqRecord(Seq(pep_seq), id=f"sp|{iso_id}|CTERM_VAR", description="")
    final_records.append(rec)

# 2. Add Internal/N-terminal Isoforms (Filtering for Status)
for _, row in sequential_results_TryP.iterrows():
    if row['Status'] == "Unique Marker Found":
        pep_seq = row['Unique_Peptide']
        iso_id = row['Isoform_ID']
        
        if pd.notna(pep_seq):
            rec = SeqRecord(Seq(pep_seq), id=f"sp|{iso_id}|INTERNAL_VAR", description="")
            final_records.append(rec)

# Write to file
with open("TryP_Isoform_Database_Final.fasta", "w") as output_handle:
    SeqIO.write(final_records, output_handle, "fasta")

print(f"FASTA created with {len(final_records)} entries.")